# Frontend Pipeline

实现 notebook 里定义的最小前端全流程，并最终生成 `macro_op_list`。


## 流程
- 初始化 `NandConfig`、`InferenceConfig`、`LlamaModelConfig`
- 初始化 `LlamaDecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass` 生成 `macro_op_list`
- 断言关键输出存在


In [ ]:
import json
from collections import Counter
from pathlib import Path

import torch
from torch.fx import GraphModule

from nandmachine.commands.macro import FlashAttnOp, MatMulOp, VectorOp
from nandmachine.config.config import NandConfig
from nandmachine.config.inference_config import InferenceConfig, ParallelConfig
from nandmachine.config.model_config import LlamaModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.llama import LlamaDecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state

MODEL_CARD_PATH = Path("model_cards/llama-405B.json")
assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH


def node_summary(graph_module: GraphModule) -> list[tuple[str, str]]:
    return [(node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
model_config = LlamaModelConfig.from_dict(json.loads(MODEL_CARD_PATH.read_text()))
nand_config = NandConfig(
    num_channels=1,
    num_plane=1,
    num_block=4,
    num_pages=16,
    tRead=1.0,
    tWrite=2.0,
    tErase=3.0,
    page_size=4,
    sram_threshold=1024,
)
inference_config = InferenceConfig(
    batch_size=2,
    input_sequence_length=8,
    output_sequence_length=4,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024,
    parallel_config=ParallelConfig(num_ranks=1),
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)

print(model_config)
print(kv_cache_state)


In [ ]:
with torch.device("meta"):
    model = LlamaDecoderLayer(model_config)
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]

print("normalized nodes:", node_summary(graph_module))
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("vector op summary:", [(op.vector_op_type, op.vector_shape) for op in vector_ops])


In [ ]:
assert macro_op_list
assert any(isinstance(op, MatMulOp) for op in macro_op_list)
assert any(isinstance(op, FlashAttnOp) for op in macro_op_list)
assert any(op.vector_op_type == "rms_norm" for op in vector_ops)
assert any(op.vector_op_type == "silu_mul" for op in vector_ops)
assert all(all(dim > 0 for dim in op.vector_shape) for op in vector_ops)

print("frontend pipeline completed successfully")


In [ ]:
print(macro_op_list)

In [ ]:
for inst in macro_op_list:
    print(inst)

In [ ]:
len(macro_op_list)

测试完整的集成工作

直接复用前面 cell 里已经生成好的 `nand_config`、`inference_config`、`kv_cache_state` 和 `macro_op_list`：
- 初始化 xPU 模拟器
- 正确加载 frontend 产出的各种 commands
- 异步运行并输出最终时间

In [ ]:
from nandmachine.simulator.entry_point import run_macro_ops

simulation_configs = [
    {"device_name": "A100_80GB", "compile_mode": "heuristic-GPU", "weight_bits": 16},
    {"device_name": "A100_80GB", "compile_mode": "heuristic-GPU", "weight_bits": 8},
    {"device_name": "H100_PCIE", "compile_mode": "heuristic-GPU", "weight_bits": 16},
    {"device_name": "H100_PCIE", "compile_mode": "heuristic-GPU", "weight_bits": 8},
    {"device_name": "H100_SXM", "compile_mode": "heuristic-GPU", "weight_bits": 16},
    {"device_name": "H100_SXM", "compile_mode": "heuristic-GPU", "weight_bits": 8},
    {"device_name": "H200_NVL", "compile_mode": "heuristic-GPU", "weight_bits": 16},
    {"device_name": "H200_NVL", "compile_mode": "heuristic-GPU", "weight_bits": 8},
    {"device_name": "H200_SXM", "compile_mode": "heuristic-GPU", "weight_bits": 16},
    {"device_name": "H200_SXM", "compile_mode": "heuristic-GPU", "weight_bits": 8},
]

simulation_results = []
for simulator_config in simulation_configs:
    final_cycle = run_macro_ops(
        nand_config,
        macro_op_list,
        **simulator_config,
    )
    assert final_cycle > 0
    simulation_results.append(
        {
            "device_name": simulator_config["device_name"],
            "precision": f"FP{simulator_config['weight_bits']}",
            "weight_bits": simulator_config["weight_bits"],
            "final_cycle": final_cycle,
        }
    )

print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("nand config:", nand_config)
print("inference config:", inference_config)
print("kv cache state:", kv_cache_state)
for result in simulation_results:
    print(result)

best_result = min(simulation_results, key=lambda item: item["final_cycle"])
print("best result:", best_result)
